In [1]:
import pickle
import torch
from tqdm import tqdm
import gzip
import pandas as pd
import os

In [2]:
def parse(path):
    g = gzip.open(path, 'rb')
    for l in g:
        yield eval(l)

In [3]:
def get_df(path):
    i = 0
    df = {}
    for d in parse(path):
        df[i] = d
        i += 1
    return pd.DataFrame.from_dict(df, orient='index')

In [4]:
def amazon(dataset):
    data_path = f'./{dataset}/reviews_{dataset}_5.json.gz'
    data_df = get_df(data_path)
    inter_df = data_df.rename(
        columns={'reviewerID': 'user', 'asin': 'item', 'unixReviewTime': 'timestamp', 'overall': 'stars'})
    inter_df = inter_df[['user', 'item', 'timestamp']]
    user_list = sorted(inter_df['user'].unique())
    user2id = dict(zip(user_list, range(1, len(user_list) + 1)))
    inter_df['user'] = inter_df['user'].apply(lambda x: user2id[x])
    return inter_df

In [5]:
def inter2txt(inter_df, txt_path):
    df = inter_df.sort_values(by=['user', 'timestamp'], kind='mergesort').reset_index(drop=True)
    with open(txt_path, 'w') as f:
        f.write('user_id:token\titem_id:token\ttimestamp:float\n')
        for i, row in tqdm(df.iterrows(), desc='Generating inter file', total=df.shape[0]):
            user, item, t = row['user'], row['item'], row['timestamp']
            f.write('{}\t{}\t{}\n'.format(user, item, t))

In [6]:
def prepare_inter(dataset):
    if not os.path.exists(dataset + '.inter'):
        inter_df = amazon(dataset)
        inter2txt(inter_df, f'{dataset}/{dataset}.inter')

In [7]:
def local_minmax_day(dataset, data_path):
    data_path = dataset + '/' + data_path
    with open(dataset + '/' + dataset + '.inter', 'r') as f:
        line = f.readlines()[1:]
    min_date, max_date = 9999999, 0
    for l in line:
        user, _, timestamp = l.strip().split('\t')
        user, timestamp = int(user), int(timestamp)
        min_date = min(int(timestamp / 86400), min_date)
        max_date = max(int(timestamp / 86400), max_date)
    with open(data_path, 'wb') as f:
        print(min_date)
        print(max_date)
        pickle.dump((min_date, max_date), f)

In [8]:
def local_timestamp(dataset, data_path):
    data_path = dataset + '/' + data_path
    with open(dataset + '/' + dataset + '.inter', 'r') as f:
        line = f.readlines()[1:]
    tmp = [-1, 0]
    max_interval = 0
    for l in line:
        user, _, timestamp = l.strip().split('\t')
        user, timestamp = int(user), int(timestamp)
        if user != tmp[0]:
            tmp = [user, timestamp]
        else:
            interval = timestamp - tmp[1]
            tmp[1] = timestamp
            max_interval = max(interval, max_interval)
    max_interval = torch.log2(torch.tensor(max_interval) + 1).item()
    with open(data_path, 'wb') as f:
        print(max_interval)
        pickle.dump(max_interval, f)

In [9]:
#dataset Amazon_Beauty
dataset = 'Electronics'
prepare_inter(dataset)
local_timestamp(dataset, 'interval_num')
local_minmax_day(dataset, 'minmax_num')

Generating inter file: 100%|██████████| 1689188/1689188 [00:29<00:00, 57673.94it/s]


28.71134376525879
10755
16274
